## BMLL Notebook Educational Series
### Futures Front Month and Top-of-Book Analysis

In [1]:
import os
from functools import lru_cache
import sys

def install_snowflake_connector():
    os.system(sys.executable + " -m pip install snowflake-connector-python[pandas] snowflake-sqlalchemy")

@lru_cache(5)
def snowflake_connection(
    database="PROD_DERIVED_DATA_DB",
    schema="PUBLIC",
    warehouse="PROD_DERIVED_DEFAULT_WH",
    role="PROD_SALES_RO_ROLE",
    user=None,
    password=None,
    use_secret_storage=True,
    sso=None,
):
    import snowflake
    import snowflake.connector
    try:
        from snowflake.snowpark import Session as SnowparkSession
    except Exception:
        SnowparkSession = None
    try:
        import bmll2
        stor = bmll2.storage_paths()["user"]
        os.environ["SNOWFLAKE_USER"] = stor["prefix"].split("/")[1]
        os.environ["SNOWFLAKE_USE_SSO"] = "1"
    except Exception:
        pass
    if sso is None:
        sso = os.environ.get("SNOWFLAKE_USE_SSO")

    password = os.environ.get("SNOWFLAKE_PASSWORD", password)
    if not password:
        try:
            import secretstorage
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            for item in collection.search_items({"name": "default_snowflake", "system": "bmll"}):
                password = item.get_secret()
        except Exception as e:
            print(f"Error connecting to Secret Service: {e}")
    else:
        try:
            import secretstorage
            attributes = {"name": "default_snowflake", "system": "bmll"}
            label = "default_snowflake"
            secret = password
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            if collection.is_locked():
                unlocked = collection.unlock()
                if unlocked:
                    print("Collection unlocked successfully.")
                else:
                    print("Failed to unlock the collection.")
            else:
                print("Collection is already unlocked.")
            collection.create_item(label, attributes, secret, replace=True)
        except Exception as e:
            print(e)

    if not sso:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            password=password,
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
        )
    else:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
            authenticator="externalbrowser",
        )
    if SnowparkSession is not None:
        return SnowparkSession.builder.configs(connargs).create()
    return snowflake.connector.connect(**connargs)

session = snowflake_connection()
if not hasattr(session, "sql"):
    class _SFResult:
        def __init__(self, cursor):
            self._cursor = cursor

        def to_pandas(self):
            try:
                return self._cursor.fetch_pandas_all()
            except Exception:
                import pandas as pd
                rows = self._cursor.fetchall()
                cols = [c[0] for c in self._cursor.description] if self._cursor.description else []
                return pd.DataFrame(rows, columns=cols)

    class _SFSessionShim:
        def __init__(self, conn):
            self._conn = conn

        def sql(self, query):
            cur = self._conn.cursor()
            cur.execute(query)
            return _SFResult(cur)

    session = _SFSessionShim(session)

/home/bertrand/miniconda3/lib/python3.11/site-packages/bmll2/_internals/api/configure.py:116: UserWarning: BMLL configuration file (/tmp/.nb_extension_instance_vars.json) not found. BMLL data and S3 storage will not be reachable.
  warnings.warn(f"{msg} BMLL data and S3 storage will not be reachable.", UserWarning)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C0433grw8&SAMLRequest=lZJdb5swFIb%2FCvKuwYZkTWaFRPlQs0zpGjV02nbnYge8GJv5mJDu18%2FkQ2ovWqkXIGTe18855z2jybFSwUFYkEanKI4ICoTODZe6SNFjdhsOUQCOac6U0SJFzwLQZDwCVqmaThtX6gfxtxHgAn%2BRBtr9SFFjNTUMJFDNKgHU5XQ7vVvTJCKUAQjrPA5dLBykZ5XO1RTjtm2jthcZW%2BCEEILJF%2BxVneQTeoGo32fU1jiTG3W1HH1PbyBiTPodwis8YXMxzqQ%2Bj%2BA9ytNZBPRrlm3Czf02Q8H02t3caGgqYbfCHmQuHh%2FW5wLAV%2FBUKRV2LyfyMgJt2p1ie5Gbqm6cvzHyX3gnOFamkH5Oq0WK6r3k38r%2BcatYqWewXN3NREKGzWG5b2a7G%2Ffvz%2FqnqeFezfiymP7KUfDjmmrSpboCaMRKd1k6f0SSm5AMQzLI4s%2BUDGivH8WD%2Fm8ULHyWUjN3cl4LZnluGu0gKowplDjVZ3AXRIIlryf%2BkTydk36vV9h2iM4LQk9MO%2F5I2yP80nnZs%2B9%2B9KvFxiiZPwe3xlbMvZ1MHMWnE8nD3UlKRcWkmnJuBYBPSCnTzq1gzq%2Bzs41AeHymvl7o8X8%3D&RelayState=ver%3A3-hint%3A10767547

In [2]:
mic = 'XCBT'
productcode = 'IQB'
start_date = '2025-01-01'
end_date = '2025-01-21'

### Why Front-Month Futures Matter

**Front month** refers to the contract with the nearest expiration date for any given product (e.g., Oil, Gold, Interest Rates).
It is critical because:

**1. Market Sentiment**
- Front month prices reflect real-time market expectations for the immediate future
- Price divergence between front and deferred months indicates market uncertainty
- Used by traders to gauge directional bias and volatility expectations

**2. Liquidity and Execution Quality**
- Front month contracts have the highest trading volume and tightest bid-ask spreads
- Easier to enter/exit positions with lower slippage
- Most cost-effective month for traders to express their views

**3. Roll Mechanics**
- Traders must monitor front-month expiry dates to plan rolling strategies
- Identifying the front month programmatically is essential for algorithmic execution
- Understanding roll periods helps predict liquidity changes and price volatility

### Top-of-Book Data

**Best Bid/Ask (Level 1)** provides real-time market prices:
- **Bid**: Highest price buyers willing to pay (demand level)
- **Ask**: Lowest price sellers willing to accept (supply level)
- **Spread**: Ask - Bid (tightness indicates liquidity quality)
- **Bid/Ask Size**: Quantity available at best prices

Top-of-book analysis reveals:
- Market microstructure dynamics (spreads, depth evolution)
- Execution cost estimation (slippage from mid-price)
- Market stress periods (widening spreads, size reduction)
- Algorithmic order impact (before/after price changes)

## Data Retrieval Functions

In [3]:
import pandas as pd 
pd.set_option('display.max_columns', None)

- **Get Top of Book for Front Month Contracts at Transaction Level**  
  The "Top of Book" provides the highest bid and ask prices, along with the quantities available at these prices. This data helps traders assess real-time market liquidity, price discovery, and potential execution costs. Analyzing top-of-book data at the transaction level is vital for detecting price slippage and market impact, ultimately enabling traders to improve execution quality and minimize trading costs.
  - **Parameters**: Exchange, Product, Date Range (Start Date and End Date)

In [4]:

def get_top_of_book(mic, productcode, start_date, end_date):
    '''
    For a given MIC and ProductCode, retrieve the front month contract, then
    retrieve the BBO for the front month contract for each date between the
    given start_date and end_date (inclusive)
    
    Input:
        mic: (str) MIC (Can accept a list)
        productcode: (str) ProductCode for a Futures product (Can accept a list)
        start_date: (str) Start of date range
        end_date: (str) End of date range
    
    Output:
        pd.DataFrame
    '''
    def _sql_list(values):
        if isinstance(values, (list, tuple, set)):
            items = list(values)
        else:
            items = [values]
        return ", ".join(f"'{v}'" for v in items)

    mic_list = _sql_list(mic)
    product_list = _sql_list(productcode)

    ref_sql = f"""
        SELECT *
        FROM "PROD_L2_CBBO_DB"."PUBLIC"."FUTURE_REFERENCE"
        WHERE MIC IN ({mic_list})
          AND PRODUCT_CODE IN ({product_list})
          AND CONTRACT_TYPE = 'Outright'
          AND REFERENCE_DATE BETWEEN '{start_date}' AND '{end_date}'
    """

    market_state_sql = f"""
        SELECT DISTINCT LISTING_ID, TRADE_DATE
        FROM "PROD_L2_CBBO_DB"."PUBLIC"."FUTURE_MARKET_STATE"
        WHERE MIC IN ({mic_list})
          AND TRADE_DATE BETWEEN '{start_date}' AND '{end_date}'
    """

    front_month_sql = f"""
        WITH ref AS ({ref_sql}),
        alive AS ({market_state_sql}),
        joined AS (
            SELECT a.TRADE_DATE, r.LISTING_ID, r.PRODUCT_CODE, r.MATURITY_MONTH_YEAR
            FROM "PROD_L2_CBBO_DB"."PUBLIC"."ALIVE" a
            INNER JOIN ref r
                ON a.LISTING_ID = r.LISTING_ID
               AND a.TRADE_DATE = r.REFERENCE_DATE
        ),
        front_month AS (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY TRADE_DATE, PRODUCT_CODE
                       ORDER BY MATURITY_MONTH_YEAR
                   ) AS "rn"
            FROM "PROD_L2_CBBO_DB"."PUBLIC"."JOINED"
        )
        SELECT * FROM "PROD_L2_CBBO_DB"."PUBLIC"."FRONT_MONTH" WHERE rn = 1
    """

    l1_sql = f"""
        SELECT
            j.PRODUCT_CODE,
            j.MATURITY_MONTH_YEAR,
            l.LISTING_ID,
            l.TRADE_DATE AS DATE,
            l.EVENT_TIMESTAMP,
            l.BID_SIZE AS BEST_BID_SIZE,
            l.BID_PRICE AS BEST_BID_PRICE,
            l.ASK_SIZE AS BEST_ASK_SIZE,
            l.ASK_PRICE AS BEST_ASK_PRICE,
            l.MARKET_STATE
        FROM ({front_month_sql}) AS "j"
        LEFT JOIN "PROD_L2_CBBO_DB"."PUBLIC"."FUTURE_L1" AS "l"
          ON l.LISTING_ID = j.LISTING_ID
         AND l.TRADE_DATE = j.TRADE_DATE
        ORDER BY l.TRADE_DATE, l.LISTING_ID, l.EVENT_TIMESTAMP
    """

    return session.sql(l1_sql)




df = get_top_of_book(mic=['XCBT', 'XEUR'],
                    productcode=['IQB', 'HYB', 'DHB', 'FUIG', 'FUHY'], 
                    start_date='2025-01-01',
                    end_date='2025-01-21'
                    )
df